# Versao 10 - Pre-Processamento

Este notebook torna explicito o que a `versao10` adiciona ao fluxo de pre-processamento. O objetivo nao e apenas preparar os dados para a rede, mas preservar sinais que o artigo considera relevantes para um cenario realista.

## Artefatos Da Ultima Execucao

Os artefatos da `versao10` existem neste diretorio, mas o ambiente usado para regenerar os notebooks nao tem `numpy` instalado para ler os arquivos `.npz`.

Depois de executar este notebook em um ambiente completo, esta secao deve voltar a mostrar:

- tamanhos dos splits;
- quantidade de variaveis e atributos tabulares;
- taxa media de `missing values`;
- taxa media de `frozen values`;
- cobertura dos rotulos por observacao (`class` e `state`).

## O que mudou em relacao a versao 9

A `versao9` ja havia melhorado a representacao temporal, mas ainda concentrava a supervisao principal no rotulo global da instancia. A `versao10` avanca em quatro frentes:

- passa a usar o conjunto completo de `27` variaveis;
- cria mascaras explicitas de ausencia e congelamento;
- reamostra tambem os rotulos sequenciais de `class` e `state`;
- registra a origem da amostra para uso posterior pela arquitetura.

## Funcoes-chave Tornadas Explicitas

Abaixo estao, em forma visivel no notebook, os blocos centrais de pre-processamento da `versao10`.

### Preenchimento de series

In [ ]:
def _fill_series(values: np.ndarray, *, discrete: bool) -> np.ndarray:
    series = pd.Series(values, dtype="float64")
    if series.notna().sum() == 0:
        return np.zeros(len(series), dtype=np.float64)
    if not discrete:
        series = series.interpolate(method="linear", limit_direction="both")
    series = series.ffill().bfill().fillna(0.0)
    return series.to_numpy(dtype=np.float64)

### Deteccao de frozen values

In [ ]:
def _compute_frozen_mask(values: np.ndarray, *, tolerance: float = 1e-12, min_run: int = 3) -> np.ndarray:
    arr = np.asarray(values, dtype=np.float64)
    if len(arr) == 0:
        return np.zeros(0, dtype=np.float64)
    frozen = np.zeros(len(arr), dtype=np.float64)
    if min_run <= 1:
        return frozen
    for end in range(min_run - 1, len(arr)):
        window = arr[end - min_run + 1 : end + 1]
        if np.isfinite(window).all() and np.max(window) - np.min(window) <= tolerance:
            frozen[end - min_run + 1 : end + 1] = 1.0
    return frozen

### Reamostragem numerica

In [ ]:
def _resample_numeric(values: np.ndarray, sequence_length: int, *, discrete: bool = False) -> np.ndarray:
    values_arr = np.asarray(values, dtype=np.float64)
    if len(values_arr) == 0:
        raise ValueError("Nao e possivel reamostrar uma sequencia vazia.")
    if len(values_arr) == 1:
        return np.repeat(values_arr, repeats=sequence_length)

    source_pos = np.linspace(0.0, 1.0, len(values_arr), dtype=np.float64)
    target_pos = np.linspace(0.0, 1.0, sequence_length, dtype=np.float64)
    if discrete:
        interpolated_idx = np.interp(target_pos, source_pos, np.arange(len(values_arr), dtype=np.float64))
        nearest_idx = np.rint(interpolated_idx).astype(np.int64)
        nearest_idx = np.clip(nearest_idx, 0, len(values_arr) - 1)
        return values_arr[nearest_idx]
    return np.interp(target_pos, source_pos, values_arr)

### Reamostragem de labels

In [ ]:
def _resample_labels(values: np.ndarray, sequence_length: int, mapping: dict[str, int]) -> np.ndarray:
    values_arr = np.asarray(values, dtype=object)
    if len(values_arr) == 0:
        raise ValueError("Nao e possivel reamostrar labels vazios.")
    if len(values_arr) == 1:
        single = values_arr[0]
        if pd.isna(single):
            return np.full(sequence_length, IGNORE_INDEX, dtype=np.int64)
        return np.full(sequence_length, mapping.get(str(int(single)), IGNORE_INDEX), dtype=np.int64)

    source_pos = np.linspace(0.0, 1.0, len(values_arr), dtype=np.float64)
    target_pos = np.linspace(0.0, 1.0, sequence_length, dtype=np.float64)
    interpolated_idx = np.interp(target_pos, source_pos, np.arange(len(values_arr), dtype=np.float64))
    nearest_idx = np.rint(interpolated_idx).astype(np.int64)
    nearest_idx = np.clip(nearest_idx, 0, len(values_arr) - 1)

    mapped = []
    for idx in nearest_idx:
        value = values_arr[idx]
        if pd.isna(value):
            mapped.append(IGNORE_INDEX)
        else:
            mapped.append(mapping.get(str(int(value)), IGNORE_INDEX))
    return np.asarray(mapped, dtype=np.int64)

### Vetor estatistico agregado

In [ ]:
def compute_statistical_feature_vector(
    sequence_array: np.ndarray,
    feature_columns: list[str],
) -> np.ndarray:
    sequence = np.asarray(sequence_array, dtype=np.float64)
    if sequence.ndim != 2:
        raise ValueError("A sequencia precisa ter duas dimensoes: [tempo, features].")

    time_axis = np.arange(sequence.shape[0], dtype=np.float64)
    feature_values = []
    for feature_idx, _ in enumerate(feature_columns):
        values = sequence[:, feature_idx]
        if len(values) > 1:
            slope = float(np.polyfit(time_axis, values, deg=1)[0])
            mean_abs_diff = float(np.abs(np.diff(values)).mean())
        else:
            slope = 0.0
            mean_abs_diff = 0.0
        feature_values.extend(
            [
                float(values.mean()),
                float(values.std()),
                float(values.min()),
                float(values.max()),
                float(np.median(values)),
                float(values[0]),
                float(values[-1]),
                slope,
                mean_abs_diff,
            ]
        )
    return np.asarray(feature_values, dtype=np.float32)

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
PROJECT_ROOT = ROOT.parent if ROOT.name == "versao10" else ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from versao10.pipeline_v10 import (
    CONTINUOUS_SENSOR_COLUMNS,
    FULL_FEATURE_COLUMNS,
    OBSERVATION_CLASS_CODES,
    OBSERVATION_STATE_CODES,
    STATE_SENSOR_COLUMNS,
    load_bundle,
    load_split_arrays,
    prepare_classification_artifacts,
)

DATASET_ROOT = PROJECT_ROOT / "3W" / "dataset"
RUN_NAME = "classificacao_v10_multitarefa"

In [2]:
artifacts = prepare_classification_artifacts(
    dataset_root=DATASET_ROOT,
    run_name=RUN_NAME,
    random_state=42,
    sequence_length=180,
)
bundle = load_bundle(artifacts.bundle_path)

train_arrays = load_split_arrays(artifacts.split_npz_paths["train"])
validation_arrays = load_split_arrays(artifacts.split_npz_paths["validation"])
test_arrays = load_split_arrays(artifacts.split_npz_paths["test"])

print("Run dir:", artifacts.run_dir)
print("Bundle path:", artifacts.bundle_path)
print("Numero de colunas selecionadas:", len(bundle.selected_columns))
print("Numero de atributos em X_tab:", len(bundle.statistical_feature_names))
print("Mapeamento de fonte:", bundle.source_mapping)

Run dir: /home/tiagoriosrocha/Desktop/lstm-w3/artifacts/reports_v10/classificacao_v10_multitarefa
Bundle path: /home/tiagoriosrocha/Desktop/lstm-w3/artifacts/reports_v10/classificacao_v10_multitarefa/bundle_v10.json
Numero de colunas selecionadas: 27
Numero de atributos em X_tab: 243
Mapeamento de fonte: {'well': 0, 'simulated': 1, 'drawn': 2}


In [3]:
resumo_arrays = pd.DataFrame(
    [
        {
            "split": "train",
            "X_seq": train_arrays["X_seq"].shape,
            "X_tab": train_arrays["X_tab"].shape,
            "X_missing": train_arrays["X_missing"].shape,
            "X_frozen": train_arrays["X_frozen"].shape,
            "y": train_arrays["y"].shape,
            "y_step_class": train_arrays["y_step_class"].shape,
            "y_step_state": train_arrays["y_step_state"].shape,
        },
        {
            "split": "validation",
            "X_seq": validation_arrays["X_seq"].shape,
            "X_tab": validation_arrays["X_tab"].shape,
            "X_missing": validation_arrays["X_missing"].shape,
            "X_frozen": validation_arrays["X_frozen"].shape,
            "y": validation_arrays["y"].shape,
            "y_step_class": validation_arrays["y_step_class"].shape,
            "y_step_state": validation_arrays["y_step_state"].shape,
        },
        {
            "split": "test",
            "X_seq": test_arrays["X_seq"].shape,
            "X_tab": test_arrays["X_tab"].shape,
            "X_missing": test_arrays["X_missing"].shape,
            "X_frozen": test_arrays["X_frozen"].shape,
            "y": test_arrays["y"].shape,
            "y_step_class": test_arrays["y_step_class"].shape,
            "y_step_state": test_arrays["y_step_state"].shape,
        },
    ]
)

display(resumo_arrays)

,split,X_seq,X_tab,X_missing,X_frozen,y,y_step_class,y_step_state
0,train,"(1559, 180, 27)","(1559, 243)","(1559, 180, 27)","(1559, 180, 27)","(1559,)","(1559, 180)","(1559, 180)"
1,validation,"(334, 180, 27)","(334, 243)","(334, 180, 27)","(334, 180, 27)","(334,)","(334, 180)","(334, 180)"
2,test,"(335, 180, 27)","(335, 243)","(335, 180, 27)","(335, 180, 27)","(335,)","(335, 180)","(335, 180)"


In [4]:
train_missing_rate = float(train_arrays["X_missing"].mean())
train_frozen_rate = float(train_arrays["X_frozen"].mean())
valid_step_class = train_arrays["y_step_class"][train_arrays["y_step_class"] >= 0]
valid_step_state = train_arrays["y_step_state"][train_arrays["y_step_state"] >= 0]

print(f"Taxa media de missing values no treino: {train_missing_rate:.4f}")
print(f"Taxa media de frozen values no treino: {train_frozen_rate:.4f}")
print("Codigos observados para class por observacao:", sorted(np.unique(valid_step_class).tolist()))
print("Codigos observados para state por observacao:", sorted(np.unique(valid_step_state).tolist()))

Taxa media de missing values no treino: 0.6813
Taxa media de frozen values no treino: 0.8466
Codigos observados para class por observacao: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
Codigos observados para state por observacao: [0, 1, 2, 3, 4, 5, 6, 7, 8]


## Leitura final

Ao final desta etapa, a `versao10` entrega ao modelo uma representacao mais rica do que nas versoes anteriores. Em vez de esconder a imperfeicao dos dados, o pipeline transforma parte dessa imperfeicao em informacao modelavel.